In [167]:
import math
import random
import numpy
import torch
import torch.nn as nn
from torch.nn import functional as F

In [168]:
batch_size = 64
block_size = 256
n_embd = 256
n_head = 4
epochs = 100000
learning_rate = 3e-4
n_layers = 6
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"using device: {device}")

using device: mps


In [169]:
with open('data/grabbed_nietzsche.txt', 'r', encoding='utf-8') as f:
    nietzshe = f.read()

In [170]:
len(nietzshe)

5812740

In [171]:
characters = sorted(set(nietzshe))
vocab_size = len(characters)
print(''.join(characters))
print(vocab_size)


 !"&'()*,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_abcdefghijklmnopqrstuvwxyz}§ÀÄÆÈÉÊÏÜàâäæçèéêîïôöùûüŒœΚΣΤάέήίαβγδεζηθικλμνξοπρςστυφχψωόύώἀἃἄἆἈἐἑἒἔἰὀὖὰὲὶὸᾶ᾽ῑῡῢῶ‐–—‘’“”…
177


In [172]:
stoi = {s:i for i, s in enumerate(characters)}
itos = {i:s for s, i in stoi.items()}
print(stoi)
print(itos)                                     # the encoders we are using are character to token encoders

encode = lambda s: [stoi[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])
encode('hi there')
decode(encode("hi there"))


{'\n': 0, ' ': 1, '!': 2, '"': 3, '&': 4, "'": 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, '/': 12, '0': 13, '1': 14, '2': 15, '3': 16, '4': 17, '5': 18, '6': 19, '7': 20, '8': 21, '9': 22, ':': 23, ';': 24, '<': 25, '=': 26, '>': 27, '?': 28, 'A': 29, 'B': 30, 'C': 31, 'D': 32, 'E': 33, 'F': 34, 'G': 35, 'H': 36, 'I': 37, 'J': 38, 'K': 39, 'L': 40, 'M': 41, 'N': 42, 'O': 43, 'P': 44, 'Q': 45, 'R': 46, 'S': 47, 'T': 48, 'U': 49, 'V': 50, 'W': 51, 'X': 52, 'Y': 53, 'Z': 54, '[': 55, '\\': 56, ']': 57, '_': 58, 'a': 59, 'b': 60, 'c': 61, 'd': 62, 'e': 63, 'f': 64, 'g': 65, 'h': 66, 'i': 67, 'j': 68, 'k': 69, 'l': 70, 'm': 71, 'n': 72, 'o': 73, 'p': 74, 'q': 75, 'r': 76, 's': 77, 't': 78, 'u': 79, 'v': 80, 'w': 81, 'x': 82, 'y': 83, 'z': 84, '}': 85, '§': 86, 'À': 87, 'Ä': 88, 'Æ': 89, 'È': 90, 'É': 91, 'Ê': 92, 'Ï': 93, 'Ü': 94, 'à': 95, 'â': 96, 'ä': 97, 'æ': 98, 'ç': 99, 'è': 100, 'é': 101, 'ê': 102, 'î': 103, 'ï': 104, 'ô': 105, 'ö': 106, 'ù': 107, 'û': 108, 'ü': 109, 'Œ': 11

'hi there'

In [173]:
tokenized_nietzshe = torch.tensor(encode(nietzshe), dtype=torch.long)       # encode the entirety of nietzshe into tokens
                                                                            

tokenized_nietzshe[:1000]

tensor([ 48,  36,  33,   1,  48,  51,  37,  40,  37,  35,  36,  48,   1,  43,
         34,   1,  48,  36,  33,   1,  37,  32,  43,  40,  47,   0,   0,  30,
         53,   0,   0,  34,  46,  37,  33,  32,  46,  37,  31,  36,   1,  42,
         37,  33,  48,  54,  47,  31,  36,  33,   0,   0,  43,  76,   9,   1,
         36,  73,  81,   1,  78,  73,   1,  44,  66,  67,  70,  73,  77,  73,
         74,  66,  67,  77,  63,   1,  81,  67,  78,  66,   1,  78,  66,  63,
          1,  36,  59,  71,  71,  63,  76,   0,   0,   0,  48,  36,  33,   1,
         29,  42,  48,  37,  31,  36,  46,  37,  47,  48,   0,   0,  58,  42,
         43,  48,  33,  47,   1,  48,  43,   1,  54,  29,  46,  29,  48,  36,
         49,  47,  48,  46,  29,   9,   1,  29,  42,  32,   1,  33,  48,  33,
         46,  42,  29,  40,   1,  46,  33,  31,  49,  46,  46,  33,  42,  31,
         33,  58,   0,   0,   0,  48,  46,  29,  42,  47,  40,  29,  48,  33,
         32,   1,  30,  53,   0,   0,  29,  42,  48,  36,  43,  

In [174]:
# split into train and test data
n = int(len(tokenized_nietzshe)*.9)
train_nietzshe = tokenized_nietzshe[:n]
test_nietzshe = tokenized_nietzshe[n:]

print(train_nietzshe.shape, test_nietzshe.shape)

torch.Size([5231466]) torch.Size([581274])


In [175]:
train_nietzshe[:block_size+1]

tensor([48, 36, 33,  1, 48, 51, 37, 40, 37, 35, 36, 48,  1, 43, 34,  1, 48, 36,
        33,  1, 37, 32, 43, 40, 47,  0,  0, 30, 53,  0,  0, 34, 46, 37, 33, 32,
        46, 37, 31, 36,  1, 42, 37, 33, 48, 54, 47, 31, 36, 33,  0,  0, 43, 76,
         9,  1, 36, 73, 81,  1, 78, 73,  1, 44, 66, 67, 70, 73, 77, 73, 74, 66,
        67, 77, 63,  1, 81, 67, 78, 66,  1, 78, 66, 63,  1, 36, 59, 71, 71, 63,
        76,  0,  0,  0, 48, 36, 33,  1, 29, 42, 48, 37, 31, 36, 46, 37, 47, 48,
         0,  0, 58, 42, 43, 48, 33, 47,  1, 48, 43,  1, 54, 29, 46, 29, 48, 36,
        49, 47, 48, 46, 29,  9,  1, 29, 42, 32,  1, 33, 48, 33, 46, 42, 29, 40,
         1, 46, 33, 31, 49, 46, 46, 33, 42, 31, 33, 58,  0,  0,  0, 48, 46, 29,
        42, 47, 40, 29, 48, 33, 32,  1, 30, 53,  0,  0, 29, 42, 48, 36, 43, 42,
        53,  1, 41, 11,  1, 40, 49, 32, 43, 50, 37, 31, 37,  0,  0,  0, 48, 66,
        63,  1, 31, 73, 71, 74, 70, 63, 78, 63,  1, 51, 73, 76, 69, 77,  1, 73,
        64,  1, 34, 76, 67, 63, 62, 76, 

In [176]:
v = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset
for i in v:
    print(i.item())

1151593
3953557
2568686
2064931
4641254
4140311
4326054
3293344
2583753
2171976
2797764
1027876
2092846
981509
2822645
1946168
4732215
1262506
3196779
189279
5212078
4540035
4589210
3785941
5191958
2777165
5146899
725679
1120978
5049970
1673718
1664268
4306727
2643990
944696
3365505
2111245
4307123
1719465
2263260
1713906
344841
4819956
785349
566066
1309566
1986634
6146
1458370
3045958
733972
3702160
2494739
580283
3848846
2224672
4563141
3394124
3255791
2328767
4611834
4472177
3838637
3705927


In [177]:

test_nietzshe = test_nietzshe.to(device)
train_nietzshe = train_nietzshe.to(device)

def generate_batch(type=None):
    xb = []
    yb = []

    if type == 'test':
        data = test_nietzshe
    else:
        data = train_nietzshe

    start_vals = torch.randint(len(data)-block_size, (batch_size, ), device=device)
    offsets = torch.arange(block_size, device=device)
    xb = data[start_vals[:, None] + offsets]
    yb = data[start_vals[:, None] + offsets + 1]
    # n = len(data)
    # start_vals = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset 
    # for start_val in start_vals:
    #     start_val = start_val.item()
    #     xb.append(data[start_val:start_val+block_size])
    #     yb.append(data[start_val+1:start_val+block_size+1])
    
    # xb = torch.stack(xb)
    # yb = torch.stack(yb)
    #return xb.to(device), yb.to(device)                          # move batches onto the active device

    return xb, yb

xb, yb = generate_batch()

print(xb, yb)

tensor([[66, 63,  1,  ..., 66, 59, 77],
        [16, 14, 11,  ..., 72,  1, 73],
        [64, 67, 61,  ...,  1, 79, 74],
        ...,
        [77, 78, 59,  ..., 70,  1, 78],
        [73, 64,  1,  ..., 67, 64, 63],
        [72,  1, 66,  ..., 32, 33, 46]], device='mps:0') tensor([[63,  1, 66,  ..., 59, 77,  0],
        [14, 11,  0,  ...,  1, 73, 64],
        [67, 61, 79,  ..., 79, 74,  1],
        ...,
        [78, 59, 72,  ...,  1, 78, 73],
        [64,  1, 77,  ..., 64, 63,  1],
        [ 1, 66, 67,  ..., 33, 46, 42]], device='mps:0')


In [178]:
class SelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size)
        self.query = nn.Linear(n_embd, head_size)
        self.value = nn.Linear(n_embd, head_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, xb):                                              # (B, T, C)
        k = self.key(xb)                                                # (B, T, head_size=n_embd/n_head)
        q = self.query(xb)                                              # (B, T, head_size)
        v = self.value(xb)                                              # (B, T, head_size)

        B, T, C = k.shape

        weights = q @ k.transpose(-2, -1) / C**0.5                      # (B, T, C) @ (B, C, T) => (B, T, T)
                                                                        # divided by n_embd root to normalize the values, bring them to one

        

        mask = torch.tril(torch.ones(T, T, device=xb.device))          # triangular mask, built on the same device as the input
        weights = qkv.masked_fill(mask == 0, float('-inf'))      
        weights = F.softmax(weights, dim=-1)                            # softmax to get probabilities (B, T, T)
        weights = self.dropout(weights)

        out = weights @ v                                               # (B, T, T) @ (B, T, head_size) => (B, T, head_size)
        return out
      

In [179]:
class MultipleSelfAttention(nn.Module):
    def __init__(self, n_head, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([SelfAttention(n_embd // n_head) for i in range(n_head)])
        self.lin_proj = nn.Linear(n_embd, n_embd)

    def forward(self, xb):
        full_attention = []

        for head in self.heads: 
            full_attention.append(head(xb))                             # (B, T, head_size)
        
        full_attention = torch.cat(full_attention, dim=-1)              # add n_head of (B, T, head_size) along last dimension = (B, T, C)
        full_attention = self.lin_proj(full_attention)                  # linear projection of (B, T, C) to get (B, T, C)

        return full_attention


In [180]:
class CompleteSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.head_size = n_embd // n_head
        # self.key = nn.Linear(n_embd, n_embd)
        # self.query = nn.Linear(n_embd, n_embd)
        # self.value = nn.Linear(n_embd, n_embd)
        self.qkv = nn.Linear(n_embd, n_embd*3)
        self.dropout = nn.Dropout(dropout)
        self.lin_proj = nn.Linear(n_embd, n_embd)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size)))

        


    def forward(self, xb, sdpa=True):
        B, T, C = xb.shape

        qkv = self.qkv(xb)

        q, k, v = qkv.split(n_embd, dim=-1)

        k = k.reshape((B, T, n_head, self.head_size)).transpose(-2, -3)                  # (B, n_head, T, head_size) 
        q = q.reshape((B, T, n_head, self.head_size)).transpose(-2, -3)                                  
        v = v.reshape((B, T, n_head, self.head_size)).transpose(-2, -3)    
        
        if sdpa:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dropout if self.training else 0.0)
        else:
            weights = q @ k.transpose(-2, -1) / self.head_size**0.5                                                  # (B, n_head, T, head_size) @ (B, n_head, head_size, T) 

            weights = weights.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
            weights = F.softmax(weights, dim=-1)
            weights = self.dropout(weights)

            out = (weights @ v)
        out = out.transpose(-2, -3).reshape((B, T, C))
        out = self.lin_proj(out)

        return out




In [181]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, xb):
        return self.ff(xb)

In [182]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        # self.full_attend = MultipleSelfAttention(n_head, n_embd)
        self.comp_full_attend = CompleteSelfAttention(n_embd, n_head)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, xb):
        attention = xb + self.comp_full_attend(self.ln1(xb))                      # (B, T, C) => 4 * self_attend(B, T, C/4) => (B, T, C)
        feed_forward = attention + self.ff(self.ln2(attention))

        return feed_forward

In [183]:
class LTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)         # a token embedding allows us to represent letters as vectors
        self.pos_embedding = nn.Embedding(block_size, n_embd)           # attention also depends on the location of a token
                                                                        # the token embedding itself will give the same value to a character at the front vs at the end
                                                                        # pos embedding handles this
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for i in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.last_layer = nn.Linear(n_embd, vocab_size)

    def forward(self, xb, targets):
        B, T = xb.shape

        tok_embd = self.token_embedding(xb)                             # (B, T, C)
        pos_embd = self.pos_embedding(torch.arange(T, device=xb.device))  # (T, C), positions built on the input's device
        x = tok_embd + pos_embd                                       # (B, T, C) + (T, C) => (B, T, C) + (B, T, C)

        out = self.blocks(x)
        logits = self.last_layer(self.ln_f(out))
        
        if targets is None: 
            loss = None
        else:
            flat_logits = logits.view(-1, vocab_size)
            flat_targets = targets.view(-1)
            loss = F.cross_entropy(flat_logits, flat_targets)
        return logits, loss
    
    def generate(self, tokens, idx):

        for i in range(tokens):
            logits, loss = self(idx[:, -block_size:], None)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_tok), dim=1)
        
        return idx


xb, yb = generate_batch()

model = LTransformer().to(device)                                   # move all model parameters onto the active device
logits, loss = model(xb, yb)
print(logits, loss)

tensor([[[ 1.4407e-01,  5.9891e-02, -1.7285e-01,  ...,  5.2461e-01,
           1.8192e+00,  8.4178e-01],
         [-5.1960e-01,  3.7687e-01, -1.1123e-01,  ..., -2.8431e-02,
          -1.7690e-01,  1.8228e-01],
         [ 4.6825e-01,  1.9404e-01,  5.8936e-01,  ..., -3.2177e-02,
          -5.6140e-01,  1.5391e-01],
         ...,
         [ 1.5630e-01,  5.6759e-01,  6.7148e-01,  ..., -2.3268e-01,
           8.9418e-01, -7.3129e-01],
         [-2.4309e-01,  6.7481e-01, -1.6947e-02,  ..., -3.7959e-01,
          -5.6136e-02,  4.5903e-03],
         [-4.5154e-01,  4.5602e-02, -9.2285e-01,  ...,  1.0072e+00,
          -8.1353e-01, -9.1661e-01]],

        [[-1.5617e-02,  1.9145e-01, -9.2280e-02,  ..., -5.4123e-01,
           1.4174e+00,  6.3207e-01],
         [-5.5073e-01,  1.1338e-01,  4.8789e-01,  ..., -2.3996e-01,
           2.0357e-01, -5.0585e-02],
         [-9.1240e-01,  1.2677e+00,  2.5647e-01,  ..., -3.8104e-01,
          -2.1390e-01,  6.2250e-01],
         ...,
         [ 5.3026e-01, -3

In [193]:
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss(iters):
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(iters)
        for k in range(iters):
            xb, yb = generate_batch(split)                  # use the split's data (train vs test)
            tlogits, tloss = model(xb, yb)
            losses[k] = tloss.item()
        out[split] = losses.mean()
    model.train()
    return out

start = time.time()
last = start
for i in range(20000):
    xb, yb = generate_batch()
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 1000 ==0:
        estimated_loss = estimate_loss(20)
        now = time.time()
        print(f"loss at epoch {i}: {estimated_loss['train']:.4f}, {estimated_loss['test']:.4f} | "
              f"total {now - start:.1f}s, +{now - last:.1f}s since last")
        last = now

loss at epoch 0: 1.1446, 1.1751 | total 2.5s, +2.5s since last
loss at epoch 1000: 1.1187, 1.1580 | total 142.2s, +139.7s since last
loss at epoch 2000: 1.1110, 1.1457 | total 281.8s, +139.6s since last
loss at epoch 3000: 1.0958, 1.1578 | total 421.5s, +139.7s since last
loss at epoch 4000: 1.0877, 1.1488 | total 561.2s, +139.6s since last
loss at epoch 5000: 1.0786, 1.1387 | total 700.2s, +139.0s since last
loss at epoch 6000: 1.0730, 1.1349 | total 838.2s, +138.0s since last
loss at epoch 7000: 1.0674, 1.1289 | total 976.3s, +138.1s since last
loss at epoch 8000: 1.0595, 1.1321 | total 1114.4s, +138.1s since last


KeyboardInterrupt: 

In [187]:
checkpoint = {
    'model_state': model.state_dict(),          # the trained weights
    'config': {                                 # so you can rebuild the exact architecture
        'vocab_size': vocab_size,
        'n_embd': n_embd,
        'n_head': n_head,
        'n_layers': n_layers,
        'block_size': block_size,
        'dropout': dropout,
    },
    'stoi': stoi,                               # char↔token maps (derived from the data)
    'itos': itos,
}
torch.save(checkpoint, 'niche_model.pt')
print('saved to niche_model.pt')

saved to niche_model.pt


In [188]:
checkpoint = torch.load('niche_model.pt', map_location=device)

# make sure the globals match what was saved before building the model
cfg = checkpoint['config']
# (set n_embd, n_head, n_layers, block_size, dropout, vocab_size to cfg's values)

model = LTransformer().to(device)
model.load_state_dict(checkpoint['model_state'])

<All keys matched successfully>

In [194]:
model.eval()
print('model loaded and in eval mode')
idx = torch.zeros((1, 1), dtype=torch.long, device=device)          # seed token, on the active device
out = model.generate(200, idx)
print(decode(out[0].tolist()))
model.train()
print('\nreturning to training mode')

model loaded and in eval mode

one individuals cannot be divined to anture to offer itself a sleep out
of evil account of something great things. In the one only speaks
when it is the same thing that mould have admired as if it wer

returning to training mode
